# tocandira, round 3: matches

Plays the matches in `MATCHES`, one after the other, with as many games in parallel as the
runtime has CPUs, and writes each match's game records and summary line to Drive under
`matches/`. Player specs are the harness's (`depth=`, `seconds=`, `name=`, UHP options);
`{fork}` is the engine built from the submodule, `cmd=ref` the 1.0.3 reference, and
`{models}` the Drive models folder (every `weights.npz` there is exported to the engine's
`weights.bin` first).

One notebook file per parallel run: save a copy per match list. Pick the runtime with the
most CPUs (v6e-1: 44, v5e-1: 24); a game keeps one core busy, so a 400 game match at
1 s/move takes about 8 minutes on v6e-1. Time controls are fair but a little noisier on
shared vCPUs than on a quiet machine; fixed depth matches are exact.

In [ ]:
MATCHES = [
    # Generation 1, 7.5M positions, outcome blend 0.9, against the best net on 3.1M; same search options.
    {"name": "l-lam09-vs-j-net5-1s-800", "games": 800,
     "a": "cmd={fork} --nnue {models}/l/lam09-s0/weights.bin --null-move-pruning --lmr --partial-iteration --killers --rfp=100,seconds=1,name=l-lam09-s0 1s",
     "b": "cmd={fork} --nnue {models}/j/n4-net5-s0/weights.bin --null-move-pruning --lmr --partial-iteration --killers --rfp=100,seconds=1,name=j-n4-net5-s0 1s"},
]
MODELS = "/content/drive/MyDrive/tocandira/models"
OUT = "/content/drive/MyDrive/tocandira/matches"

import os
print("cpus:", os.cpu_count())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then curl -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null; fi
source "$HOME/.cargo/env"
cd /content
# Cloned to /content/repo, not /content/tocandira: a folder named like the package would shadow it on sys.path.
if [ -d repo ]; then (cd repo && git pull -q && git submodule update --init -q); else git clone -q --recurse-submodules https://github.com/abstratovcm/tocandira repo; fi
cd /content/repo/engine/nokamute
CARGO_PROFILE_RELEASE_LTO=true CARGO_PROFILE_RELEASE_CODEGEN_UNITS=1 cargo build --release -q
cd /content/repo && pip install -q .
./engine/nokamute/target/release/nokamute --help | head -1
# The frozen opponent, upstream v1.0.3.
[ -x reference/nokamute-1.0.3 ] || scripts/build_reference.sh

In [ ]:
import glob
import subprocess

FORK = "/content/repo/engine/nokamute/target/release/nokamute"
os.environ["TOCANDIRA_ENGINE"] = FORK
os.environ["TOCANDIRA_REFERENCE"] = "/content/repo/reference/nokamute-1.0.3"
for npz in glob.glob(f"{MODELS}/**/weights.npz", recursive=True):
    if not os.path.exists(npz.replace("weights.npz", "weights.bin")):
        if subprocess.run(["hive-export", npz]).returncode != 0:
            print("could not export, skipping:", npz)
print("engine weights:", glob.glob(f"{MODELS}/**/weights.bin", recursive=True))

In [ ]:
import time


def workers_for(m):
    """One game per vCPU at fixed depth (exact anyway); one per two vCPUs on the clock, so
    the two engines of a game do not share a core with another game's."""
    timed = "seconds=" in m["a"] or "seconds=" in m["b"]
    return max(1, (os.cpu_count() or 2) // (2 if timed else 1))


for m in MATCHES:
    args = ["hive-tournament", "--a", m["a"].format(fork=FORK, models=MODELS),
            "--b", m["b"].format(fork=FORK, models=MODELS), "--games", str(m["games"]),
            "--workers", str(m.get("workers", workers_for(m))), "--seed", str(m.get("seed", 1)),
            "--out", f"{OUT}/{m['name']}.jsonl"]
    t0 = time.time()
    run = subprocess.run(args, capture_output=True, text=True)
    tail = [line for line in run.stderr.splitlines() if not line.startswith("[")]
    summary = "\n".join(tail[-4:])
    print(f"=== {m['name']} ({time.time() - t0:.0f}s)\n{summary}\n")
    with open(f"{OUT}/summary.txt", "a") as f:
        f.write(f"{m['name']}\n{summary}\n\n")

Each summary is from A's point of view: W/D/L, score with its 95% interval and the implied
Elo. `Disagree` games (the two engines differ on whether a repetition ended the game, a
rules difference between the fork and 1.0.3) count as draws.